# Caso de uso Apache Spark – ShopNow (E‑commerce)

En este notebook desarrollamos paso a paso un caso de uso de Apache Spark para una empresa ficticia de e‑commerce llamada **ShopNow**.
El objetivo es mostrar, de forma pedagógica, cómo se puede diseñar una solución de recomendaciones usando Spark en la nube.

## 1. Contexto y problema

1. ShopNow registra millones de eventos de navegación y compras cada mes.
2. Sus sistemas tradicionales no permiten analizar datos en tiempo casi real.
3. El objetivo es construir un motor de recomendaciones escalable usando Apache Spark.

**Idea clave:** Spark permite procesar grandes volúmenes de datos en memoria y en clúster, combinando batch + streaming.

## 2. Tipos de datos y módulos de Spark

- Datos de compras (estructurados).
- Eventos de navegación (JSON, semiestructurados).
- Catálogo de productos.

Usaremos tres módulos principales de Spark:
1. Spark SQL / DataFrames.
2. Spark MLlib (modelo ALS).
3. Spark Streaming para eventos en tiempo casi real.

In [1]:
# Diccionarios ilustrativos para entender las fuentes de datos

fuentes_datos = {
    'compras': {
        'tipo': 'estructurado',
        'descripcion': 'Tabla con historiales de compra por cliente y producto'
    },
    'navegacion': {
        'tipo': 'semiestructurado',
        'descripcion': 'Eventos JSON con vistas, clicks, add-to-cart, etc.'
    },
    'catalogo_productos': {
        'tipo': 'estructurado',
        'descripcion': 'Información de productos: categoría, marca, precio'
    }
}

for nombre, info in fuentes_datos.items():
    print('Fuente: {} -> Tipo: {} | {}'.format(nombre, info['tipo'], info['descripcion']))

Fuente: compras -> Tipo: estructurado | Tabla con historiales de compra por cliente y producto
Fuente: navegacion -> Tipo: semiestructurado | Eventos JSON con vistas, clicks, add-to-cart, etc.
Fuente: catalogo_productos -> Tipo: estructurado | Información de productos: categoría, marca, precio


## 3. Arquitectura lógica (data lake en la nube)

Dividimos el data lake en tres zonas: **raw**, **clean/silver** y **curated/gold**.
Cada zona representa un nivel distinto de calidad y transformación de los datos.

In [2]:
# Representación simple de la estructura de un data lake para ShopNow

data_lake = {
    'raw': {
        'navegacion': 's3://shopnow-datalake/raw/navegacion/',
        'compras': 's3://shopnow-datalake/raw/compras/'
    },
    'clean': {
        'navegacion': 's3://shopnow-datalake/clean/navegacion/',
        'compras': 's3://shopnow-datalake/clean/compras/'
    },
    'curated': {
        'recomendaciones': 's3://shopnow-datalake/curated/recomendaciones/'
    }
}

for zona, rutas in data_lake.items():
    print('Zona {}:'.format(zona))
    for nombre, ruta in rutas.items():
        print('  - {}: {}'.format(nombre, ruta))

Zona raw:
  - navegacion: s3://shopnow-datalake/raw/navegacion/
  - compras: s3://shopnow-datalake/raw/compras/
Zona clean:
  - navegacion: s3://shopnow-datalake/clean/navegacion/
  - compras: s3://shopnow-datalake/clean/compras/
Zona curated:
  - recomendaciones: s3://shopnow-datalake/curated/recomendaciones/


## 4. Pseudocódigo PySpark para procesamiento batch (modelo ALS)

En esta sección mostramos un **pseudocódigo** de cómo se podría entrenar un modelo de recomendación con ALS en PySpark.
No está pensado para ejecutarse aquí, sino como guía conceptual.

In [3]:
spark_code_als = '''
from pyspark.sql import SparkSession
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator

spark = SparkSession.builder.appName('ShopNowRecs').getOrCreate()

# 1. Leer datos de compras
compras = spark.read.parquet('s3://shopnow-datalake/clean/compras/')

# Suponemos columnas: id_cliente, id_producto, rating (por ejemplo, frecuencia o preferencia)
# 2. Dividir en train/test
train, test = compras.randomSplit([0.8, 0.2], seed=42)

# 3. Definir el modelo ALS
als = ALS(
    maxIter=10,
    regParam=0.1,
    userCol='id_cliente',
    itemCol='id_producto',
    ratingCol='rating',
    coldStartStrategy='drop'
)

# 4. Entrenar el modelo
modelo = als.fit(train)

# 5. Evaluar en test
predicciones = modelo.transform(test)
evaluator = RegressionEvaluator(metricName='rmse', labelCol='rating', predictionCol='prediction')
rmse = evaluator.evaluate(predicciones)
print('RMSE del modelo ALS:', rmse)

# 6. Generar recomendaciones para cada cliente
recs_clientes = modelo.recommendForAllUsers(numItems=10)
recs_clientes.write.mode('overwrite').parquet('s3://shopnow-datalake/curated/recomendaciones/')

spark.stop()
'''

print(spark_code_als)


from pyspark.sql import SparkSession
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator

spark = SparkSession.builder.appName('ShopNowRecs').getOrCreate()

# 1. Leer datos de compras
compras = spark.read.parquet('s3://shopnow-datalake/clean/compras/')

# Suponemos columnas: id_cliente, id_producto, rating (por ejemplo, frecuencia o preferencia)
# 2. Dividir en train/test
train, test = compras.randomSplit([0.8, 0.2], seed=42)

# 3. Definir el modelo ALS
als = ALS(
    maxIter=10,
    regParam=0.1,
    userCol='id_cliente',
    itemCol='id_producto',
    ratingCol='rating',
    coldStartStrategy='drop'
)

# 4. Entrenar el modelo
modelo = als.fit(train)

# 5. Evaluar en test
predicciones = modelo.transform(test)
evaluator = RegressionEvaluator(metricName='rmse', labelCol='rating', predictionCol='prediction')
rmse = evaluator.evaluate(predicciones)
print('RMSE del modelo ALS:', rmse)

# 6. Generar recomendaciones para cada cliente
recs_clientes

## 5. Pseudocódigo PySpark para Spark Streaming + Kafka

Aquí mostramos cómo Spark Streaming podría leer eventos de navegación desde Kafka y calcular métricas en ventanas de tiempo.

In [4]:
spark_code_streaming = '''
from pyspark.sql import SparkSession
from pyspark.sql.functions import from_json, col, window
from pyspark.sql.types import StructType, StructField, StringType, TimestampType

spark = SparkSession.builder.appName('ShopNowStreaming').getOrCreate()

# 1. Leer eventos desde Kafka
df_raw = spark.readStream.format('kafka')\
    .option('kafka.bootstrap.servers', 'kafka:9092')\
    .option('subscribe', 'eventos_navegacion')\
    .load()

# 2. Esquema de los eventos
schema = StructType([
    StructField('id_cliente', StringType()),
    StructField('id_producto', StringType()),
    StructField('evento', StringType()),
    StructField('timestamp', TimestampType())
])

df_parsed = df_raw.select(
    from_json(col('value').cast('string'), schema).alias('data')
).select('data.*')

# 3. Calcular productos más vistos en ventanas de 15 minutos
df_agg = df_parsed.groupBy(
    window(col('timestamp'), '15 minutes'),
    col('id_producto')
).count().orderBy(col('count').desc())

# 4. Escribir resultados en consola o en un sink
query = df_agg.writeStream.outputMode('complete')\
    .format('console')\
    .start()

query.awaitTermination()
'''

print(spark_code_streaming)


from pyspark.sql import SparkSession
from pyspark.sql.functions import from_json, col, window
from pyspark.sql.types import StructType, StructField, StringType, TimestampType

spark = SparkSession.builder.appName('ShopNowStreaming').getOrCreate()

# 1. Leer eventos desde Kafka
df_raw = spark.readStream.format('kafka')    .option('kafka.bootstrap.servers', 'kafka:9092')    .option('subscribe', 'eventos_navegacion')    .load()

# 2. Esquema de los eventos
schema = StructType([
    StructField('id_cliente', StringType()),
    StructField('id_producto', StringType()),
    StructField('evento', StringType()),
    StructField('timestamp', TimestampType())
])

df_parsed = df_raw.select(
    from_json(col('value').cast('string'), schema).alias('data')
).select('data.*')

# 3. Calcular productos más vistos en ventanas de 15 minutos
df_agg = df_parsed.groupBy(
    window(col('timestamp'), '15 minutes'),
    col('id_producto')
).count().orderBy(col('count').desc())

# 4. Escribir resultados en c

## 6. Ventajas, desafíos y cierre

- Ventajas: menor latencia, ecosistema unificado, escalabilidad en la nube.
- Desafíos: curva de aprendizaje, calidad de datos, operación en producción, costos.

Este caso te sirve como plantilla para ejercicios del bootcamp sobre Spark y arquitecturas Big Data en e‑commerce.